# 58. DINOv3 핵심 아이디어와 모델 스케일

            DINOv3의 핵심은 더 큰 모델을 더 큰 unlabeled image dataset으로 학습하되, image-level feature뿐 아니라 patch-level dense feature 품질을 유지하는 데 있습니다.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path("Deeplearning") / "Vision 기초" / "8장",
    Path("Vision 기초") / "8장",
]
NOTEBOOK_DIR = next((p for p in candidates if (p / "seg8_utils.py").exists()), Path.cwd())
sys.path.append(str(NOTEBOOK_DIR))

from seg8_utils import *
set_korean_font()
set_seed(7)

DATA_ROOT = ensure_dataset()
RUNS_ROOT = NOTEBOOK_DIR / "runs"


## 58-1. 공개 모델 계열 정리


In [ ]:
model_family = [
    {"family": "ViT-S/16 distilled", "params_m": 21, "pretrain": "LVD-1689M web image"},
    {"family": "ViT-B/16 distilled", "params_m": 86, "pretrain": "LVD-1689M web image"},
    {"family": "ViT-L/16 distilled", "params_m": 300, "pretrain": "LVD-1689M web image"},
    {"family": "ViT-H+/16 distilled", "params_m": 840, "pretrain": "LVD-1689M web image"},
    {"family": "ViT-7B/16", "params_m": 6716, "pretrain": "LVD-1689M web image"},
    {"family": "ConvNeXt Tiny", "params_m": 29, "pretrain": "LVD-1689M web image"},
    {"family": "ConvNeXt Large", "params_m": 198, "pretrain": "LVD-1689M web image"},
    {"family": "ViT-L/16 satellite", "params_m": 300, "pretrain": "SAT-493M satellite image"},
]
model_family


## 58-2. 모델 크기 비교


In [ ]:
import matplotlib.pyplot as plt

labels = [m["family"] for m in model_family]
values = [m["params_m"] for m in model_family]
plt.figure(figsize=(10, 4))
plt.bar(labels, values)
plt.ylabel("parameters (M)")
plt.xticks(rotation=30, ha="right")
plt.yscale("log")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 58-3. 7장과의 연결

            7장의 TinyFCN/TinyUNet은 전체 모델을 직접 학습했습니다. DINOv3 기반 접근은 backbone을 대부분 고정하고 작은 adapter만 학습합니다. 그래서 비교할 때는 mean IoU뿐 아니라 학습한 parameter 수, annotation 요구량, GPU memory, 추론 시간도 함께 봐야 합니다.
